In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

In [ ]:
tickers = []

In [ ]:
class efficientFrontier:
    def __init__(self,
                 tickers
                 ):
        self.tickers = tickers
        # download data from yfinance
        prices = yf.download(self.tickers, period="2y", interval="1d")["Close"]

        self.returns = prices.pct_change().dropna()
        self.mean_returns = self.returns.mean()
        self.covariance_matrix = self.returns.cov()
        self.correlation_matrix = self.returns.corr()

    def monte_carlo(self, n_portfolios: int = 200000):
        results = []

        for _ in range(n_portfolios):
            weights = np.random.dirichlet(len(self.tickers))

            portfolio_return = np.sum(self.mean_returns * weights) * 252
            portfolio_volatility = np.sqrt(
                np.dot(weights.T, np.dot(self.covariance_matrix * 252, weights))
            )
            sharpe_ratio = portfolio_return / portfolio_volatility \
                if portfolio_volatility != 0 else 0
            
            results.append([portfolio_return, portfolio_volatility, sharpe_ratio] + list(weights))

        cols = ["Return", "Volatility", "Sharpe Ratio"] + self.tickers
        self.portfolios = pd.DataFrame(results, columns=cols)

        self.max_sharpe = self.portfolios.loc[self.portfolios["Sharpe Ratio"].idxmax()]
        self.min_volatility = self.portfolios.loc[self.portfolios["Volatility"].idxmin()]
    
    def regression(self):
        raise NotImplementedError
    
    def plot_monte_carlo(self):
        raise NotImplementedError
    
    def plot_regression(self):
        raise NotImplementedError
    
    def plot_correlation_matrix(self):
        raise NotImplementedError